# 05 — OOF evaluation & residual analysis

**Variant:** `typewell_gr_alignment` · Error analysis by depth and residual distribution (competition post-train diagnostics).


In [ ]:
"""Shared imports for Rogii TVT pipeline notebooks."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
if not (NB_DIR / "phase_runner.py").is_file():
    NB_DIR = Path(r"/lustre/work/sweeden/agent-tracing-trace-baseline/examples/rogii/traces/preprocessing/typewell_gr_alignment/notebooks")
VARIANT_DIR = NB_DIR.parent
ROGII_ROOT = Path("/lustre/work/sweeden/rogii")
sys.path.insert(0, str(NB_DIR))
if str(ROGII_ROOT) not in sys.path:
    sys.path.insert(0, str(ROGII_ROOT))

from phase_runner import (
    ARTIFACTS_ROOT,
    TRACE_INDEX,
    require_prior_phase,
    load_phase_manifest,
    trace_steps_for_phase,
    _resolve_path,
    _read_json,
    _load_train_predict,
)

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

PHASE = "05_evaluation"


## 1. Load training predictions


In [ ]:
require_prior_phase(PHASE)
p01 = load_phase_manifest("01_data_analysis")
p04 = load_phase_manifest("04_model_training")
paths = _read_json(_resolve_path(p01["outputs"]["data_paths"]))
transform = _read_json(_resolve_path(p04["outputs"]["transform"]))
target = transform["target_column"]
train_df = pd.read_csv(paths["train_csv"])
tp = _load_train_predict(); train_df = tp.align_train_target_to_schema(train_df, target)
oof = np.load(_resolve_path(p04["outputs"]["oof_predictions"]))
y_true = train_df[target].astype(float).values


## 2. Recompute OOF RMSE


In [ ]:
if transform.get("use_log1p"):
    pred = np.expm1(np.clip(oof, None, 20))
else:
    pred = oof
mask = np.isfinite(pred) & np.isfinite(y_true)
rmse = tp.rmse(y_true[mask], pred[mask])
print("OOF RMSE:", rmse)


## 3. Residual distribution


In [ ]:
residual = y_true[mask] - pred[mask]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(residual, bins=40, color="crimson", alpha=0.85)
axes[0].set_title("Residual histogram")
axes[1].scatter(pred[mask], residual, s=4, alpha=0.3)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel("y_pred"); axes[1].set_ylabel("residual")
plt.tight_layout(); plt.show()


## 4. Residuals vs measured depth


In [ ]:
if "MD" in train_df.columns:
    md = train_df.loc[mask, "MD"].values
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.scatter(md, np.abs(residual), s=4, alpha=0.25)
    ax.set_xlabel("MD"); ax.set_ylabel("|residual|"); ax.set_title("Absolute error vs depth")
    plt.tight_layout(); plt.show()


## 5. Predicted vs true (OOF)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_true[mask], pred[mask], s=4, alpha=0.25)
lims = [min(y_true[mask].min(), pred[mask].min()), max(y_true[mask].max(), pred[mask].max())]
ax.plot(lims, lims, "k--", lw=0.8)
ax.set_xlabel("y_true"); ax.set_ylabel("y_pred")
plt.tight_layout(); plt.show()


## 6. Error by target quantile bin


In [ ]:
bins = pd.qcut(y_true[mask], q=5, duplicates="drop")
err_by_bin = pd.DataFrame({"y": y_true[mask], "abs_err": np.abs(residual), "bin": bins})
err_by_bin.groupby("bin", observed=True)["abs_err"].mean()


## 7. Well-level error (if group key available)


In [ ]:
from pipeline import well_group_detector as wgd
id_col = _read_json(_resolve_path(p01["outputs"]["schema"]))["id_column"]
gk = wgd.recommend_group_key(train_df, None, id_column=id_col)
if gk and gk in train_df.columns:
    tmp = train_df.loc[mask, [gk]].copy()
    tmp["abs_err"] = np.abs(residual)
    print(tmp.groupby(gk)["abs_err"].mean().sort_values(ascending=False).head())
else:
    print("No well group column for per-well error breakdown")


## 8. Competition diagnostic note

Top Rogii writeups emphasize **depth-stratified** and **leakage-aware** error views — this phase mirrors that post-train analysis before submission.


## 9. RMSE vs MAE summary


In [ ]:
mae = np.mean(np.abs(residual))
print({"rmse": float(rmse), "mae": float(mae), "n_oof": int(mask.sum())})


## 10. Residual skewness


In [ ]:
from scipy.stats import skew
print("residual skew:", skew(residual))


## 11. Mean absolute error by quantile


In [ ]:
abs_err = np.abs(residual)
pd.Series(abs_err).describe()


## 12. Residual autocorrelation along index


In [ ]:
if len(residual) > 10:
    ac1 = np.corrcoef(residual[:-1], residual[1:])[0, 1]
    print("lag-1 residual autocorr:", ac1)


## 13. Persist evaluation artifacts


In [ ]:
from phase_runner import run_05_evaluation
manifest = run_05_evaluation()
print(json.dumps(manifest, indent=2))


## 14. Metrics JSON


In [ ]:
metrics = _read_json(_resolve_path(manifest["outputs"]["metrics"]))
print(json.dumps(metrics, indent=2))


## 15. Residuals CSV head


In [ ]:
res_path = _resolve_path(manifest["outputs"]["residuals"])
display(pd.read_csv(res_path).head())


## 16. Depth-binned error summary


In [ ]:
depth_path = ARTIFACTS_ROOT / PHASE / "residuals_by_depth.json"
if depth_path.is_file():
    print(json.dumps(_read_json(depth_path), indent=2))


## 17. Q-Q plot (normality check on residuals)


In [ ]:
import scipy.stats as stats
fig, ax = plt.subplots(figsize=(4, 4))
stats.probplot(residual, dist="norm", plot=ax)
ax.set_title("Residual Q-Q")
plt.tight_layout(); plt.show()


## 18. Trace coverage


In [ ]:
print("trace steps:", len(trace_steps_for_phase(PHASE)))


## 19. Evaluation artifact checklist


In [ ]:
for name in ["metrics.json", "residuals.csv", "residuals_by_depth.json", "phase_manifest.json"]:
    p = ARTIFACTS_ROOT / PHASE / name
    print(name, "OK" if p.is_file() else "MISSING")


## 20. Handoff to submission

Phase 06 validates the submission CSV against `sample_submission.csv` row order and column schema.
